# Read dictionary values in the ERD payload

Follow one dictionary through the actual Maxitor query.

Run in a fresh kernel using the repository's Python environment. The matching script is `09_erd.py`.


## Prepare the model

These declarations supply the complete model needed by this case. Run this cell before the experiment.


In [1]:
from pydantic import Field

from aoa.action_machine.domain import BaseCoreEntity, BaseEnumEntity
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.entity import entity


class ShopDomain(BaseDomain):
    """Group the shop declarations."""

    name = "shop"
    description = "Shop model"


@entity(description="Currencies accepted by the shop", domain=ShopDomain)
class CurrencyEnumEntity(BaseEnumEntity):
    """Declare the shop's currency vocabulary."""

    symbol: str = Field(description="Currency symbol")

    EUR = CurrencyEnumEntity(code="eur", display_name="Euro", symbol="€")  # noqa: F821
    USD = CurrencyEnumEntity(code="usd", display_name="US dollar", symbol="$")  # noqa: F821


@entity(description="Price tag", domain=ShopDomain)
class PriceTagCoreEntity(BaseCoreEntity):
    """Associate an amount with its currency."""

    id: str = Field(description="Tag identifier")
    amount: float = Field(description="Price amount")
    currency: CurrencyEnumEntity = Field(description="Money unit")


## Run the experiment

Follow one dictionary through the actual Maxitor query.


In [2]:
import json

from aoa.action_machine.runtime.action_product_machine import ActionProductMachine
from aoa.maxitor.model.diagrams.actions.list_entities_action import ListEntitiesAction
from aoa.maxitor.model.diagrams.resources.duckdb_graph_resource import DuckDBGraphResource

machine = ActionProductMachine(loggers=[])
domain = next(node.node_id for node in machine.graph_coordinator.get_all_nodes() if node.label == "ShopDomain")
store = DuckDBGraphResource.build_from_json(json.loads(machine.graph_coordinator.to_json()))
diagram = ListEntitiesAction._slice_payload(store, domain, include_neighbors=False)
currency = next(item for item in diagram["entities"] if item["label"] == "CurrencyEnumEntity")
tag = next(item for item in diagram["entities"] if item["label"] == "PriceTagCoreEntity")
print(currency["values"][0])
print(next(field["reference_qualname"] for field in tag["fields"] if field["name"] == "currency").split(".")[-1])


{'code': 'eur', 'display_name': 'Euro', 'properties': [{'name': 'symbol', 'value': '€'}]}
CurrencyEnumEntity


## Explain the result

The diagram query preserves values and their descriptive fields. Its presentation payload expresses properties as name/value rows of text, which the card renders below the label. The full graph retains the original properties object. _slice_payload is an internal inspection API, used here to reproduce the viewer without an HTTP service.
